# Notebook 31 — Cross-Script Transformation Feasibility Audit

## Objective

Notebook 29 and Notebook 30 showed that explicit script invariance is not a reliable objective for cross-script writer verification.

The experiments revealed:

- script information is strongly entangled with writer geometry,
- adversarial classifier confusion does not guarantee nuisance removal,
- removing script-related directions may also remove writer-discriminative information.

Therefore, this notebook investigates an alternative hypothesis:

> Instead of removing script variation, can we model the writer-preserving transformation caused by script change?

The goal is not to train the final verification model.

This notebook only performs a feasibility audit.

---

## Hypothesis

For a writer \(w\):

Arabic representation:

\[
z_A^w
\]

English representation:

\[
z_E^w
\]

may be related through a cross-script transformation:

\[
T(z_A^w) \approx z_E^w
\]

where \(T\) captures the systematic change caused by writing in another script while preserving writer-specific characteristics.

If such a transformation exists:

- same-writer Arabic→English prediction error should be lower than random-writer prediction error,
- the learned transformation should generalize to unseen writers.

---

## Dataset Structure

Each writer has four pages:

- p1: Arabic variable
- p2: Arabic fixed
- p3: English variable
- p4: English fixed

For this audit:

Arabic representation:

\[
A_w = \frac{p1_w+p2_w}{2}
\]

English representation:

\[
E_w = \frac{p3_w+p4_w}{2}
\]

are used as script-balanced writer centroids.

This averaging reduces text-condition imbalance.

---

## Representation

The audit uses:

1. Raw cached DINOv2-S 384-D features

2. Notebook 27 standard writer projection:

\[
384 \rightarrow 144
\]

No parameters are trained in the representation.

---

## Experimental Protocol

Fit writers:

- 145 writers

Selection writers:

- 36 unseen writers

No writer overlap exists.

A transformation is fitted only on fit writers.

The transformation is then evaluated on selection writers.

---

## Transformation Models

This notebook evaluates simple feasibility baselines.

### 1. Identity baseline

Measures the natural distance:

\[
||A_w-E_w||
\]

before any transformation.

### 2. Linear transformation

Learn:

\[
W A_w \approx E_w
\]

using fit writers only.

No nonlinear model is used.

The purpose is to test whether a simple shared cross-script mapping exists.

---

## Evaluation

For each selection writer:

Same writer error:

\[
||T(A_w)-E_w||
\]

Random writer error:

\[
||T(A_i)-E_j||
\]

where:

\[
i \neq j
\]

A useful transformation should produce:

\[
same\_writer\_error
<
random\_writer\_error
\]

with a meaningful margin.

---

## Interpretation Boundary

A successful transformation does not prove a final verification method.

It only supports the hypothesis that cross-script writer correspondence has predictable geometry.

A failed transformation means this direction should not be developed further.

No hyperparameter tuning or model selection will be performed in this notebook.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [5]:
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent


SOURCE_ARTIFACT_PATH = (
    ROOT
    / "reports"
    / "paired_writer_conditioned_script_geometry_audit"
    / "paired_script_geometry_aligned_representations.npz"
)

ROLE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "development_internal_writer_roles_seed42.csv"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "cross_script_transformation_feasibility_audit"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


required_paths = {
    "aligned_representation_artifact": SOURCE_ARTIFACT_PATH,
    "writer_roles": ROLE_PATH,
}


missing_paths = {
    name: str(path)
    for name, path in required_paths.items()
    if not path.exists()
}


if len(missing_paths) > 0:
    raise FileNotFoundError(
        json.dumps(
            missing_paths,
            indent=2,
        )
    )


role_df = pd.read_csv(
    ROLE_PATH
)


artifact = np.load(
    SOURCE_ARTIFACT_PATH,
    allow_pickle=False,
)


required_keys = {
    "fit_writer_ids",
    "selection_writer_ids",
    "fit_raw_by_writer",
    "selection_raw_by_writer",
    "fit_projected_by_writer",
    "selection_projected_by_writer",
    "fit_page_ids",
    "selection_page_ids",
    "fit_filenames",
    "selection_filenames",
}


if not required_keys.issubset(
    set(
        artifact.files
    )
):
    raise RuntimeError(
        "Required artifact keys are missing."
    )


fit_writer_ids = (
    artifact[
        "fit_writer_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

selection_writer_ids = (
    artifact[
        "selection_writer_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)


fit_raw_by_writer = (
    artifact[
        "fit_raw_by_writer"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

selection_raw_by_writer = (
    artifact[
        "selection_raw_by_writer"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)


fit_projected_by_writer = (
    artifact[
        "fit_projected_by_writer"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

selection_projected_by_writer = (
    artifact[
        "selection_projected_by_writer"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)


fit_page_ids = (
    artifact[
        "fit_page_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

selection_page_ids = (
    artifact[
        "selection_page_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)


fit_filenames = (
    artifact[
        "fit_filenames"
    ]
    .astype(str)
)

selection_filenames = (
    artifact[
        "selection_filenames"
    ]
    .astype(str)
)


fit_selection_overlap = (
    set(
        fit_writer_ids.tolist()
    )
    &
    set(
        selection_writer_ids.tolist()
    )
)


expected_fit_pages = np.tile(
    np.array(
        [
            1,
            2,
            3,
            4,
        ],
        dtype=np.int64,
    ),
    (
        len(
            fit_writer_ids
        ),
        1,
    ),
)


expected_selection_pages = np.tile(
    np.array(
        [
            1,
            2,
            3,
            4,
        ],
        dtype=np.int64,
    ),
    (
        len(
            selection_writer_ids
        ),
        1,
    ),
)


fit_page_order_valid = bool(
    np.array_equal(
        fit_page_ids,
        expected_fit_pages,
    )
)


selection_page_order_valid = bool(
    np.array_equal(
        selection_page_ids,
        expected_selection_pages,
    )
)


fit_raw_norms = np.linalg.norm(
    fit_raw_by_writer.reshape(
        -1,
        fit_raw_by_writer.shape[-1],
    ),
    axis=1,
)

selection_raw_norms = np.linalg.norm(
    selection_raw_by_writer.reshape(
        -1,
        selection_raw_by_writer.shape[-1],
    ),
    axis=1,
)


fit_projected_norms = np.linalg.norm(
    fit_projected_by_writer.reshape(
        -1,
        fit_projected_by_writer.shape[-1],
    ),
    axis=1,
)

selection_projected_norms = np.linalg.norm(
    selection_projected_by_writer.reshape(
        -1,
        selection_projected_by_writer.shape[-1],
    ),
    axis=1,
)


transformation_resource_audit = {
    "notebook": 31,
    "experiment": (
        "cross-script transformation feasibility audit"
    ),
    "source_artifact": str(
        SOURCE_ARTIFACT_PATH.relative_to(
            ROOT
        )
    ),
    "fit_writer_count": int(
        len(
            fit_writer_ids
        )
    ),
    "selection_writer_count": int(
        len(
            selection_writer_ids
        )
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_selection_overlap
        )
    ),
    "fit_raw_shape": list(
        fit_raw_by_writer.shape
    ),
    "selection_raw_shape": list(
        selection_raw_by_writer.shape
    ),
    "fit_projected_shape": list(
        fit_projected_by_writer.shape
    ),
    "selection_projected_shape": list(
        selection_projected_by_writer.shape
    ),
    "raw_feature_dimension": int(
        fit_raw_by_writer.shape[-1]
    ),
    "projected_feature_dimension": int(
        fit_projected_by_writer.shape[-1]
    ),
    "fit_page_order_valid": fit_page_order_valid,
    "selection_page_order_valid": selection_page_order_valid,
    "fit_raw_unit_normalized": bool(
        np.allclose(
            fit_raw_norms,
            1.0,
            atol=1e-5,
            rtol=0.0,
        )
    ),
    "selection_raw_unit_normalized": bool(
        np.allclose(
            selection_raw_norms,
            1.0,
            atol=1e-5,
            rtol=0.0,
        )
    ),
    "fit_projected_unit_normalized": bool(
        np.allclose(
            fit_projected_norms,
            1.0,
            atol=1e-5,
            rtol=0.0,
        )
    ),
    "selection_projected_unit_normalized": bool(
        np.allclose(
            selection_projected_norms,
            1.0,
            atol=1e-5,
            rtol=0.0,
        )
    ),
    "parameters_trained": False,
    "transformation_fitted": False,
    "selection_used_for_training": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "transformation_resource_audit.json",
    "w",
) as file:
    json.dump(
        transformation_resource_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        transformation_resource_audit,
        indent=2,
    )
)


if (
    fit_raw_by_writer.shape
    != (
        145,
        4,
        384,
    )
    or selection_raw_by_writer.shape
    != (
        36,
        4,
        384,
    )
    or fit_projected_by_writer.shape
    != (
        145,
        4,
        144,
    )
    or selection_projected_by_writer.shape
    != (
        36,
        4,
        144,
    )
    or len(
        fit_selection_overlap
    )
    != 0
    or not fit_page_order_valid
    or not selection_page_order_valid
    or not np.allclose(
        fit_raw_norms,
        1.0,
        atol=1e-5,
        rtol=0.0,
    )
    or not np.allclose(
        selection_raw_norms,
        1.0,
        atol=1e-5,
        rtol=0.0,
    )
    or not np.allclose(
        fit_projected_norms,
        1.0,
        atol=1e-5,
        rtol=0.0,
    )
    or not np.allclose(
        selection_projected_norms,
        1.0,
        atol=1e-5,
        rtol=0.0,
    )
):
    raise RuntimeError(
        "Notebook 31 resource audit failed."
    )

{
  "notebook": 31,
  "experiment": "cross-script transformation feasibility audit",
  "source_artifact": "reports/paired_writer_conditioned_script_geometry_audit/paired_script_geometry_aligned_representations.npz",
  "fit_writer_count": 145,
  "selection_writer_count": 36,
  "fit_selection_writer_overlap": 0,
  "fit_raw_shape": [
    145,
    4,
    384
  ],
  "selection_raw_shape": [
    36,
    4,
    384
  ],
  "fit_projected_shape": [
    145,
    4,
    144
  ],
  "selection_projected_shape": [
    36,
    4,
    144
  ],
  "raw_feature_dimension": 384,
  "projected_feature_dimension": 144,
  "fit_page_order_valid": true,
  "selection_page_order_valid": true,
  "fit_raw_unit_normalized": true,
  "selection_raw_unit_normalized": true,
  "fit_projected_unit_normalized": true,
  "selection_projected_unit_normalized": true,
  "parameters_trained": false,
  "transformation_fitted": false,
  "selection_used_for_training": false,
  "monitor_used": false,
  "validation_used": false,
  "o

In [6]:
def build_script_centroids(
    writer_page_features,
):
    arabic_centroids = (
        writer_page_features[
            :,
            0:2,
            :
        ]
        .mean(
            axis=1
        )
    )

    english_centroids = (
        writer_page_features[
            :,
            2:4,
            :
        ]
        .mean(
            axis=1
        )
    )

    return {
        "arabic": arabic_centroids,
        "english": english_centroids,
    }


def euclidean_distance(
    left,
    right,
):
    return np.linalg.norm(
        left - right,
        axis=1,
    )


def cosine_distance(
    left,
    right,
):
    left_norm = (
        left
        /
        np.linalg.norm(
            left,
            axis=1,
            keepdims=True,
        )
    )

    right_norm = (
        right
        /
        np.linalg.norm(
            right,
            axis=1,
            keepdims=True,
        )
    )

    return 1.0 - np.sum(
        left_norm
        * right_norm,
        axis=1,
    )


def summarize_array(
    values,
):
    return {
        "count": int(
            len(
                values
            )
        ),
        "mean": float(
            np.mean(
                values
            )
        ),
        "std": float(
            np.std(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.median(
                values
            )
        ),
        "minimum": float(
            np.min(
                values
            )
        ),
        "maximum": float(
            np.max(
                values
            )
        ),
    }


fit_raw_centroids = (
    build_script_centroids(
        fit_raw_by_writer
    )
)

selection_raw_centroids = (
    build_script_centroids(
        selection_raw_by_writer
    )
)


fit_projected_centroids = (
    build_script_centroids(
        fit_projected_by_writer
    )
)

selection_projected_centroids = (
    build_script_centroids(
        selection_projected_by_writer
    )
)


fit_raw_same_euclidean = (
    euclidean_distance(
        fit_raw_centroids[
            "arabic"
        ],
        fit_raw_centroids[
            "english"
        ],
    )
)

fit_raw_same_cosine = (
    cosine_distance(
        fit_raw_centroids[
            "arabic"
        ],
        fit_raw_centroids[
            "english"
        ],
    )
)


fit_projected_same_euclidean = (
    euclidean_distance(
        fit_projected_centroids[
            "arabic"
        ],
        fit_projected_centroids[
            "english"
        ],
    )
)

fit_projected_same_cosine = (
    cosine_distance(
        fit_projected_centroids[
            "arabic"
        ],
        fit_projected_centroids[
            "english"
        ],
    )
)


selection_raw_same_euclidean = (
    euclidean_distance(
        selection_raw_centroids[
            "arabic"
        ],
        selection_raw_centroids[
            "english"
        ],
    )
)

selection_projected_same_euclidean = (
    euclidean_distance(
        selection_projected_centroids[
            "arabic"
        ],
        selection_projected_centroids[
            "english"
        ],
    )
)


rng = np.random.default_rng(
    42
)


def random_writer_distances(
    arabic_centroids,
    english_centroids,
    trials=500,
):
    rows = []

    count = len(
        arabic_centroids
    )

    for trial in range(
        trials
    ):
        permutation = (
            rng.permutation(
                count
            )
        )

        distances = (
            euclidean_distance(
                arabic_centroids,
                english_centroids[
                    permutation
                ],
            )
        )

        rows.append(
            {
                "trial": trial,
                "mean_distance": float(
                    np.mean(
                        distances
                    )
                ),
                "median_distance": float(
                    np.median(
                        distances
                    )
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


fit_raw_random_df = (
    random_writer_distances(
        fit_raw_centroids[
            "arabic"
        ],
        fit_raw_centroids[
            "english"
        ],
    )
)

fit_projected_random_df = (
    random_writer_distances(
        fit_projected_centroids[
            "arabic"
        ],
        fit_projected_centroids[
            "english"
        ],
    )
)


identity_baseline_summary = {
    "notebook": 31,
    "experiment": (
        "cross-script transformation feasibility"
    ),
    "analysis_stage": (
        "identity baseline only"
    ),
    "fit_writers": 145,
    "selection_writers": 36,
    "raw_fit_same_writer_euclidean": (
        summarize_array(
            fit_raw_same_euclidean
        )
    ),
    "raw_fit_same_writer_cosine_distance": (
        summarize_array(
            fit_raw_same_cosine
        )
    ),
    "projected_fit_same_writer_euclidean": (
        summarize_array(
            fit_projected_same_euclidean
        )
    ),
    "projected_fit_same_writer_cosine_distance": (
        summarize_array(
            fit_projected_same_cosine
        )
    ),
    "selection_raw_same_writer_euclidean": (
        summarize_array(
            selection_raw_same_euclidean
        )
    ),
    "selection_projected_same_writer_euclidean": (
        summarize_array(
            selection_projected_same_euclidean
        )
    ),
    "raw_random_writer_mean_distance": (
        summarize_array(
            fit_raw_random_df[
                "mean_distance"
            ]
        )
    ),
    "projected_random_writer_mean_distance": (
        summarize_array(
            fit_projected_random_df[
                "mean_distance"
            ]
        )
    ),
    "transformation_fitted": False,
    "parameters_trained": False,
    "selection_used_for_training": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "identity_baseline_summary.json",
    "w",
) as file:
    json.dump(
        identity_baseline_summary,
        file,
        indent=2,
    )


print(
    json.dumps(
        identity_baseline_summary,
        indent=2,
    )
)


if (
    fit_raw_centroids[
        "arabic"
    ].shape
    != (
        145,
        384,
    )
    or fit_projected_centroids[
        "arabic"
    ].shape
    != (
        145,
        144,
    )
    or selection_raw_centroids[
        "arabic"
    ].shape
    != (
        36,
        384,
    )
):
    raise RuntimeError(
        "Centroid construction failed."
    )

{
  "notebook": 31,
  "experiment": "cross-script transformation feasibility",
  "analysis_stage": "identity baseline only",
  "fit_writers": 145,
  "selection_writers": 36,
  "raw_fit_same_writer_euclidean": {
    "count": 145,
    "mean": 0.3654205799102783,
    "std": 0.06795777380466461,
    "median": 0.3612196147441864,
    "minimum": 0.231377512216568,
    "maximum": 0.6508241295814514
  },
  "raw_fit_same_writer_cosine_distance": {
    "count": 145,
    "mean": 0.07295209914445877,
    "std": 0.02928853966295719,
    "median": 0.06826436519622803,
    "minimum": 0.02764129638671875,
    "maximum": 0.22830677032470703
  },
  "projected_fit_same_writer_euclidean": {
    "count": 145,
    "mean": 0.5408067107200623,
    "std": 0.12036058306694031,
    "median": 0.52228844165802,
    "minimum": 0.25518545508384705,
    "maximum": 0.8432358503341675
  },
  "projected_fit_same_writer_cosine_distance": {
    "count": 145,
    "mean": 0.16661840677261353,
    "std": 0.0736904963850975,


In [7]:
from sklearn.linear_model import Ridge


TRANSFORMATION_ALPHA = 1.0
RANDOM_PAIR_TRIALS = 500


def fit_linear_transformation(
    source,
    target,
    alpha,
):
    model = Ridge(
        alpha=alpha,
        fit_intercept=True,
        random_state=42,
    )

    model.fit(
        source,
        target,
    )

    return model


def evaluate_transformation(
    source,
    target,
    transformation,
    random_trials=500,
):
    predicted = transformation.predict(
        source
    )

    same_euclidean = np.linalg.norm(
        predicted - target,
        axis=1,
    )

    same_cosine = 1.0 - (
        np.sum(
            (
                predicted
                /
                np.linalg.norm(
                    predicted,
                    axis=1,
                    keepdims=True,
                )
            )
            *
            (
                target
                /
                np.linalg.norm(
                    target,
                    axis=1,
                    keepdims=True,
                )
            ),
            axis=1,
        )
    )

    rng = np.random.default_rng(
        42
    )

    random_rows = []

    count = len(
        source
    )

    for trial in range(
        random_trials
    ):
        permutation = (
            rng.permutation(
                count
            )
        )

        random_target = (
            target[
                permutation
            ]
        )

        random_error = np.linalg.norm(
            predicted - random_target,
            axis=1,
        )

        random_rows.append(
            {
                "trial": trial,
                "mean_error": float(
                    random_error.mean()
                ),
                "median_error": float(
                    np.median(
                        random_error
                    )
                ),
            }
        )

    random_df = pd.DataFrame(
        random_rows
    )

    return {
        "predicted": predicted,
        "same_euclidean": same_euclidean,
        "same_cosine": same_cosine,
        "random_df": random_df,
    }


def summarize_values(
    values,
):
    return {
        "count": int(
            len(
                values
            )
        ),
        "mean": float(
            np.mean(
                values
            )
        ),
        "std": float(
            np.std(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.median(
                values
            )
        ),
        "minimum": float(
            np.min(
                values
            )
        ),
        "maximum": float(
            np.max(
                values
            )
        ),
    }


def run_transformation_experiment(
    fit_source,
    fit_target,
    selection_source,
    selection_target,
    name,
):
    transformation = fit_linear_transformation(
        fit_source,
        fit_target,
        TRANSFORMATION_ALPHA,
    )

    fit_result = evaluate_transformation(
        fit_source,
        fit_target,
        transformation,
        RANDOM_PAIR_TRIALS,
    )

    selection_result = evaluate_transformation(
        selection_source,
        selection_target,
        transformation,
        RANDOM_PAIR_TRIALS,
    )

    return {
        "name": name,
        "model": transformation,
        "fit": fit_result,
        "selection": selection_result,
    }


raw_transformation_result = (
    run_transformation_experiment(
        fit_raw_centroids[
            "arabic"
        ],
        fit_raw_centroids[
            "english"
        ],
        selection_raw_centroids[
            "arabic"
        ],
        selection_raw_centroids[
            "english"
        ],
        "raw_dinov2s_384d",
    )
)


projected_transformation_result = (
    run_transformation_experiment(
        fit_projected_centroids[
            "arabic"
        ],
        fit_projected_centroids[
            "english"
        ],
        selection_projected_centroids[
            "arabic"
        ],
        selection_projected_centroids[
            "english"
        ],
        "notebook27_projection_144d",
    )
)


transformation_summary_rows = []


for result in [
    raw_transformation_result,
    projected_transformation_result,
]:
    for split_name in [
        "fit",
        "selection",
    ]:
        split_result = result[
            split_name
        ]

        transformation_summary_rows.append(
            {
                "representation": (
                    result[
                        "name"
                    ]
                ),
                "split": split_name,
                "same_writer_euclidean_mean": float(
                    split_result[
                        "same_euclidean"
                    ].mean()
                ),
                "same_writer_euclidean_median": float(
                    np.median(
                        split_result[
                            "same_euclidean"
                        ]
                    )
                ),
                "same_writer_cosine_mean": float(
                    split_result[
                        "same_cosine"
                    ].mean()
                ),
                "random_writer_mean_error_mean": float(
                    split_result[
                        "random_df"
                    ][
                        "mean_error"
                    ].mean()
                ),
                "random_writer_mean_error_std": float(
                    split_result[
                        "random_df"
                    ][
                        "mean_error"
                    ].std(
                        ddof=1
                    )
                ),
                "random_gap": float(
                    split_result[
                        "random_df"
                    ][
                        "mean_error"
                    ].mean()
                    -
                    split_result[
                        "same_euclidean"
                    ].mean()
                ),
            }
        )


transformation_summary_df = pd.DataFrame(
    transformation_summary_rows
)


RAW_TRANSFORMATION_PATH = (
    REPORT_DIR
    / "linear_cross_script_transformations.npz"
)


np.savez_compressed(
    RAW_TRANSFORMATION_PATH,
    raw_fit_predictions=(
        raw_transformation_result[
            "fit"
        ][
            "predicted"
        ]
    ),
    raw_selection_predictions=(
        raw_transformation_result[
            "selection"
        ][
            "predicted"
        ]
    ),
    projected_fit_predictions=(
        projected_transformation_result[
            "fit"
        ][
            "predicted"
        ]
    ),
    projected_selection_predictions=(
        projected_transformation_result[
            "selection"
        ][
            "predicted"
        ]
    ),
)


transformation_audit = {
    "notebook": 31,
    "experiment": (
        "linear cross-script transformation feasibility"
    ),
    "transformation": (
        "Ridge regression Arabic centroid -> English centroid"
    ),
    "alpha": float(
        TRANSFORMATION_ALPHA
    ),
    "fit_writers": 145,
    "selection_writers": 36,
    "raw_transformation_fit": {
        "same_writer_euclidean": (
            summarize_values(
                raw_transformation_result[
                    "fit"
                ][
                    "same_euclidean"
                ]
            )
        ),
        "random_writer_error": (
            summarize_values(
                raw_transformation_result[
                    "fit"
                ][
                    "random_df"
                ][
                    "mean_error"
                ]
            )
        ),
    },
    "raw_transformation_selection": {
        "same_writer_euclidean": (
            summarize_values(
                raw_transformation_result[
                    "selection"
                ][
                    "same_euclidean"
                ]
            )
        ),
        "random_writer_error": (
            summarize_values(
                raw_transformation_result[
                    "selection"
                ][
                    "random_df"
                ][
                    "mean_error"
                ]
            )
        ),
    },
    "projected_transformation_fit": {
        "same_writer_euclidean": (
            summarize_values(
                projected_transformation_result[
                    "fit"
                ][
                    "same_euclidean"
                ]
            )
        ),
        "random_writer_error": (
            summarize_values(
                projected_transformation_result[
                    "fit"
                ][
                    "random_df"
                ][
                    "mean_error"
                ]
            )
        ),
    },
    "projected_transformation_selection": {
        "same_writer_euclidean": (
            summarize_values(
                projected_transformation_result[
                    "selection"
                ][
                    "same_euclidean"
                ]
            )
        ),
        "random_writer_error": (
            summarize_values(
                projected_transformation_result[
                    "selection"
                ][
                    "random_df"
                ][
                    "mean_error"
                ]
            )
        ),
    },
    "transformation_fitted": True,
    "selection_used_for_training": False,
    "hyperparameter_tuned_on_selection": False,
    "parameters_trained": True,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "linear_transformation_feasibility_audit.json",
    "w",
) as file:
    json.dump(
        transformation_audit,
        file,
        indent=2,
    )


transformation_summary_df.to_csv(
    REPORT_DIR
    / "linear_transformation_summary.csv",
    index=False,
)


print(
    json.dumps(
        transformation_audit,
        indent=2,
    )
)

print(
    "\nTransformation summary:"
)

print(
    transformation_summary_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        transformation_summary_df
    )
    != 4
    or not np.isfinite(
        transformation_summary_df[
            [
                "same_writer_euclidean_mean",
                "random_writer_mean_error_mean",
                "random_gap",
            ]
        ]
        .to_numpy()
    ).all()
):
    raise RuntimeError(
        "Transformation feasibility audit failed."
    )

{
  "notebook": 31,
  "experiment": "linear cross-script transformation feasibility",
  "transformation": "Ridge regression Arabic centroid -> English centroid",
  "alpha": 1.0,
  "fit_writers": 145,
  "selection_writers": 36,
  "raw_transformation_fit": {
    "same_writer_euclidean": {
      "count": 145,
      "mean": 0.23133011162281036,
      "std": 0.05802283436059952,
      "median": 0.2231743484735489,
      "minimum": 0.13344696164131165,
      "maximum": 0.5354443192481995
    },
    "random_writer_error": {
      "count": 500,
      "mean": 0.27012346041202545,
      "std": 0.0020762675706401264,
      "median": 0.27012138068675995,
      "minimum": 0.26364865899086,
      "maximum": 0.27546441555023193
    }
  },
  "raw_transformation_selection": {
    "same_writer_euclidean": {
      "count": 36,
      "mean": 0.26174551248550415,
      "std": 0.047621726989746094,
      "median": 0.25698715448379517,
      "minimum": 0.19007490575313568,
      "maximum": 0.4087297320365906

In [9]:
from sklearn.metrics import roc_auc_score, roc_curve


def cosine_similarity_matrix(
    left,
    right,
):
    left_norm = (
        left
        /
        np.linalg.norm(
            left,
            axis=1,
            keepdims=True,
        )
    )

    right_norm = (
        right
        /
        np.linalg.norm(
            right,
            axis=1,
            keepdims=True,
        )
    )

    return (
        left_norm
        @
        right_norm.T
    )


def compute_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr

    index = np.nanargmin(
        np.abs(
            fpr - fnr
        )
    )

    return float(
        (
            fpr[index]
            +
            fnr[index]
        )
        / 2.0
    )


def build_page_level_pairs(
    arabic_pages,
    english_pages,
    arabic_writer_ids,
    english_writer_ids,
):
    similarity = cosine_similarity_matrix(
        arabic_pages,
        english_pages,
    )

    scores = []
    labels = []

    for i in range(
        len(
            arabic_writer_ids
        )
    ):
        for j in range(
            len(
                english_writer_ids
            )
        ):
            scores.append(
                similarity[
                    i,
                    j,
                ]
            )

            labels.append(
                int(
                    arabic_writer_ids[i]
                    ==
                    english_writer_ids[j]
                )
            )

    return (
        np.asarray(
            scores,
            dtype=np.float32,
        ),
        np.asarray(
            labels,
            dtype=np.int64,
        ),
    )


def evaluate_pair_scores(
    scores,
    labels,
):
    return {
        "pairs": int(
            len(
                labels
            )
        ),
        "genuine_pairs": int(
            labels.sum()
        ),
        "impostor_pairs": int(
            (
                labels == 0
            ).sum()
        ),
        "auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),
        "eer": float(
            compute_eer(
                labels,
                scores,
            )
        ),
    }


def evaluate_transformation_verification(
    arabic_pages,
    english_pages,
    arabic_writer_ids,
    english_writer_ids,
    transformation,
):
    transformed_arabic = (
        transformation.predict(
            arabic_pages
        )
    )

    baseline_scores, labels = (
        build_page_level_pairs(
            arabic_pages,
            english_pages,
            arabic_writer_ids,
            english_writer_ids,
        )
    )

    transformed_scores, _ = (
        build_page_level_pairs(
            transformed_arabic,
            english_pages,
            arabic_writer_ids,
            english_writer_ids,
        )
    )

    return {
        "baseline": (
            evaluate_pair_scores(
                baseline_scores,
                labels,
            )
        ),
        "transformed": (
            evaluate_pair_scores(
                transformed_scores,
                labels,
            )
        ),
    }


# ======================================
# Build page-level Arabic / English data
# ======================================

fit_raw_arabic_pages = (
    fit_raw_by_writer[
        :,
        0:2,
        :
    ]
    .reshape(
        -1,
        384,
    )
)

fit_raw_english_pages = (
    fit_raw_by_writer[
        :,
        2:4,
        :
    ]
    .reshape(
        -1,
        384,
    )
)


fit_projected_arabic_pages = (
    fit_projected_by_writer[
        :,
        0:2,
        :
    ]
    .reshape(
        -1,
        144,
    )
)

fit_projected_english_pages = (
    fit_projected_by_writer[
        :,
        2:4,
        :
    ]
    .reshape(
        -1,
        144,
    )
)


selection_raw_arabic_pages = (
    selection_raw_by_writer[
        :,
        0:2,
        :
    ]
    .reshape(
        -1,
        384,
    )
)

selection_raw_english_pages = (
    selection_raw_by_writer[
        :,
        2:4,
        :
    ]
    .reshape(
        -1,
        384,
    )
)


selection_projected_arabic_pages = (
    selection_projected_by_writer[
        :,
        0:2,
        :
    ]
    .reshape(
        -1,
        144,
    )
)

selection_projected_english_pages = (
    selection_projected_by_writer[
        :,
        2:4,
        :
    ]
    .reshape(
        -1,
        144,
    )
)


# Each writer has 2 Arabic pages and 2 English pages
fit_page_writer_ids = np.repeat(
    fit_writer_ids,
    2,
)

selection_page_writer_ids = np.repeat(
    selection_writer_ids,
    2,
)


# ======================================
# Fit transformations on writer centroid
# ======================================

raw_page_transformation = (
    fit_linear_transformation(
        fit_raw_centroids[
            "arabic"
        ],
        fit_raw_centroids[
            "english"
        ],
        TRANSFORMATION_ALPHA,
    )
)


projected_page_transformation = (
    fit_linear_transformation(
        fit_projected_centroids[
            "arabic"
        ],
        fit_projected_centroids[
            "english"
        ],
        TRANSFORMATION_ALPHA,
    )
)


# ======================================
# Selection evaluation only
# ======================================

raw_selection_page_result = (
    evaluate_transformation_verification(
        selection_raw_arabic_pages,
        selection_raw_english_pages,
        selection_page_writer_ids,
        selection_page_writer_ids,
        raw_page_transformation,
    )
)


projected_selection_page_result = (
    evaluate_transformation_verification(
        selection_projected_arabic_pages,
        selection_projected_english_pages,
        selection_page_writer_ids,
        selection_page_writer_ids,
        projected_page_transformation,
    )
)


page_level_transformation_audit = {
    "notebook": 31,
    "experiment": (
        "page-level cross-script transformation verification feasibility"
    ),
    "fit_writers": 145,
    "selection_writers": 36,
    "transformation_training": (
        "fit writer centroid Arabic -> English"
    ),
    "selection_used_for_training": False,
    "raw_selection": (
        raw_selection_page_result
    ),
    "projected_selection": (
        projected_selection_page_result
    ),
    "transformation_fitted": True,
    "verification_evaluated": True,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "page_level_transformation_verification_audit.json",
    "w",
) as file:
    json.dump(
        page_level_transformation_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        page_level_transformation_audit,
        indent=2,
    )
)


if (
    raw_selection_page_result[
        "baseline"
    ][
        "pairs"
    ]
    != 5184
    or projected_selection_page_result[
        "baseline"
    ][
        "pairs"
    ]
    != 5184
    or raw_selection_page_result[
        "baseline"
    ][
        "genuine_pairs"
    ]
    != 144
    or projected_selection_page_result[
        "baseline"
    ][
        "genuine_pairs"
    ]
    != 144
):
    raise RuntimeError(
        "Page-level pair construction failed."
    )

{
  "notebook": 31,
  "experiment": "page-level cross-script transformation verification feasibility",
  "fit_writers": 145,
  "selection_writers": 36,
  "transformation_training": "fit writer centroid Arabic -> English",
  "selection_used_for_training": false,
  "raw_selection": {
    "baseline": {
      "pairs": 5184,
      "genuine_pairs": 144,
      "impostor_pairs": 5040,
      "auc": 0.6634610615079365,
      "eer": 0.3729166666666667
    },
    "transformed": {
      "pairs": 5184,
      "genuine_pairs": 144,
      "impostor_pairs": 5040,
      "auc": 0.5909019510582011,
      "eer": 0.43878968253968254
    }
  },
  "projected_selection": {
    "baseline": {
      "pairs": 5184,
      "genuine_pairs": 144,
      "impostor_pairs": 5040,
      "auc": 0.8372946979717814,
      "eer": 0.24295634920634923
    },
    "transformed": {
      "pairs": 5184,
      "genuine_pairs": 144,
      "impostor_pairs": 5040,
      "auc": 0.8143959435626102,
      "eer": 0.25555555555555554
    }
  

# Final Summary

Notebook 31 investigated whether cross-script writer verification could be reformulated as a transformation problem rather than a script-invariance problem.

The central hypothesis was:

> Instead of removing script information, can an Arabic writer representation be transformed into the corresponding English writer representation while preserving writer identity?

The notebook was intentionally designed as a feasibility audit rather than a final model-development experiment.

---

## Experimental Protocol

Two frozen representations were studied:

1. Raw cached DINOv2-S features
   - 384 dimensions

2. Notebook 27 writer projection
   - 384 → 144 dimensions
   - L2-normalized embeddings

The clean development protocol was preserved:

- Fit writers: 145
- Selection writers: 36
- Writer overlap: 0

No monitor, validation, or official-test writers were used.

The transformation was fitted only on the 145 fit writers.

---

# Finding 1 — Cross-Script Writer Correspondence Exists Before Transformation

For each writer, script-level centroids were constructed:

\[
A_w=\frac{p1_w+p2_w}{2}
\]

and

\[
E_w=\frac{p3_w+p4_w}{2}
\]

where \(A_w\) is the Arabic centroid and \(E_w\) is the English centroid.

Before learning any transformation, same-writer Arabic-English centroids were already closer than random-writer pairings.

For raw DINOv2-S:

- Fit same-writer distance: 0.3654
- Random-writer distance: 0.4285

The same-writer distance remained almost unchanged on unseen selection writers:

- Selection same-writer distance: 0.3667

For the Notebook 27 projection:

- Fit same-writer distance: 0.5408
- Random-writer distance: 0.9696

This confirms that both representations already contain cross-script writer correspondence.

---

# Finding 2 — A Linear Arabic-to-English Transformation Is Learnable at the Writer-Centroid Level

A fixed Ridge regression transformation was fitted:

\[
T(A_w)\approx E_w
\]

using only the 145 fit writers.

No transformation hyperparameter was selected using the selection writers.

For raw DINOv2-S:

Before transformation:

\[
0.3654
\]

After transformation on fit writers:

\[
0.2313
\]

On unseen selection writers:

\[
0.3667 \rightarrow 0.2617
\]

Therefore, the learned mapping generalized beyond the writers used to fit it.

For the Notebook 27 projection, the transformation also reduced centroid-level error.

Selection writers showed:

- Same-writer transformed error: 0.5301
- Random-writer transformed error: 0.7847

These results demonstrate that cross-script writer centroids contain a predictable shared relationship.

---

# Finding 3 — Centroid Prediction Does Not Translate Into Better Page-Level Verification

The decisive experiment evaluated the learned transformation on actual page-level cross-script verification.

Each selection split evaluation contained:

- 72 Arabic pages
- 72 English pages
- 5,184 total cross-script pairs
- 144 genuine pairs
- 5,040 impostor pairs

The transformation remained exactly the one fitted from fit-writer centroids.

No page-level transformation training was performed.

## Raw DINOv2-S

Direct baseline:

- AUC: 0.6635
- EER: 37.29%

After Arabic-to-English transformation:

- AUC: 0.5909
- EER: 43.88%

Performance became substantially worse.

## Notebook 27 Projection

Direct baseline:

- AUC: 0.8373
- EER: 24.30%

After transformation:

- AUC: 0.8144
- EER: 25.56%

Performance again declined.

Therefore:

> Reducing centroid-level Arabic-to-English reconstruction error does not imply improved cross-script writer discrimination.

---

# Scientific Interpretation

Notebook 31 reveals an important distinction.

A cross-script relationship is predictable at the writer-prototype level, but page-level verification requires preserving discriminative variation that is not captured by a single shared Arabic-to-English transformation.

The global transformation appears to make representations more predictable while simultaneously removing or distorting information useful for distinguishing writers.

Therefore, the task should not be treated as direct feature translation.

The hypothesis:

> A shared Arabic-to-English embedding transformation can directly improve cross-script writer verification

is not supported.

---

# Relation to Previous Findings

The result is consistent with the progression of the preceding experiments.

Notebook 29 showed that explicit script-adversarial invariance did not produce reliable script suppression.

Notebook 30 showed that script-related geometry overlaps strongly with writer geometry and cannot be safely treated as a removable low-dimensional nuisance.

Notebook 31 now shows that directly translating one script representation into another also does not preserve the discriminative structure required for verification.

Together, these findings suggest that cross-script writer verification is unlikely to be solved by either:

1. removing script information, or
2. translating one script embedding into another.

A more appropriate future direction is to investigate the relationship between two script-dependent writer representations directly rather than forcing them into the same representation through invariance or translation.

This is a research hypothesis for future investigation, not a novelty claim established by this notebook.

---

# Final Decision

Global cross-script transformation:

- Centroid-level feasibility: Supported
- Generalization to unseen writer centroids: Supported
- Page-level verification improvement: Not supported
- Raw DINOv2-S verification improvement: No
- Notebook 27 projection improvement: No

Therefore:

- No transformation hyperparameter tuning will be performed.
- No nonlinear transformation will be attempted in this notebook.
- No reverse or cycle-consistent transformation will be attempted.
- No monitor evaluation will be performed.
- The global transformation direction is closed.

The next research question should focus on cross-script writer correspondence rather than embedding translation.

---

# Reproducibility Status

- Fit writers: 145
- Selection writers: 36
- Selection used for transformation fitting: No
- Hyperparameters tuned on selection: No
- Page-level verification evaluated: Yes
- Monitor used: No
- Validation used: No
- Official test used: No

Notebook 31 is closed with a negative but scientifically informative result.